# Шаг 7. Маленький чат-бот

**Qwen2.5-0.5B-Instruct** (Alibaba, 2024) — одна из самых маленьких моделей, с которой можно разговаривать.
Она устроена так же, как GPT: угадывает следующий токен. Но её обучали в **два этапа**:

1. **Предобучение** — угадывать следующий токен на триллионах токенов текста (как ruGPT-3 в шаге 6, только текста в сотни раз больше).
   После этого модель умеет продолжать текст, но не умеет «отвечать на вопросы».
2. **Дообучение на диалогах** (instruct) — на тысячах примеров вида «вопрос → хороший ответ». После этого модель
   продолжает текст **диалога** репликой помощника.

Модель в 600 раз больше нашей мини-GPT и в 4 раза больше ruGPT-3, но в 1000 раз меньше самых больших чат-ботов.
Поэтому она быстро работает даже на процессоре, но часто ошибается — и это хорошо видно.

In [1]:
# Настройка для Google Colab. На своём компьютере эта ячейка ничего не делает.
# В Colab нужна папка machine_vision на Google Диске (в ней session05 и session06).
import os
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive:
    drive.mount("/content/drive")
    COLAB_ROOT = "/content/drive/MyDrive/machine_vision"
    os.chdir(f"{COLAB_ROOT}/session06")  # здесь лежит common.py; модели скачаются в machine_vision/data/hf
    sys.path.insert(0, os.getcwd())
    !pip install -q transformers

In [2]:
import json
import time

import torch

from common import load_genres, load_model, load_tokenizer

tok = load_tokenizer("chat")
model = load_model("chat", "lm")
torch.manual_seed(0)

## 1. Что такое «чат» на самом деле

Диалог — это список сообщений с ролями: `system` (инструкция для модели), `user` (человек), `assistant` (модель).
Перед тем как отдать диалог модели, его склеивают в **один текст** по шаблону (`apply_chat_template`).
Посмотрим, как выглядит этот текст.

In [3]:
messages = [
    {"role": "system", "content": "Ты — вежливый помощник школьника. Отвечай кратко."},
    {"role": "user", "content": "Что такое фотосинтез?"},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)
print("токенов в подсказке:", len(tok(prompt)["input_ids"]))

<|im_start|>system
Ты — вежливый помощник школьника. Отвечай кратко.<|im_end|>
<|im_start|>user
Что такое фотосинтез?<|im_end|>
<|im_start|>assistant

токенов в подсказке: 41


Видите служебные токены `<|im_start|>` и `<|im_end|>` (шаг 1)? Они размечают, где начинается и кончается каждая реплика.
Шаблон заканчивается на `<|im_start|>assistant` — то есть «сейчас говорит помощник». Модели остаётся только **продолжить текст**,
как делала ruGPT-3 в шаге 6. Никакой отдельной «программы для ответов» нет — только угадывание следующего токена.

Модель сама решает, когда закончить: она выдаёт токен `<|im_end|>`, и генерация останавливается.

In [4]:
def chat(messages, max_new_tokens=200, temperature=None, show_time=True, model=model, tok=tok):
    """Ответ модели на диалог. temperature=None — жадный выбор (одинаковый ответ при каждом запуске)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt")["input_ids"]
    if temperature:     # случайный выбор: температура и top-p, как в шаге 6
        how = {"do_sample": True, "temperature": temperature, "top_p": 0.9, "top_k": None}
    else:               # жадный выбор (у Qwen по умолчанию включён случайный — выключаем явно)
        how = {"do_sample": False, "temperature": None, "top_p": None, "top_k": None}
    start = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **how)
    answer = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    if show_time:
        n = out.shape[1] - ids.shape[1]
        print(f"[{n} токенов за {time.time() - start:.1f} с]")
    return answer


print(chat(messages))

[35 токенов за 2.5 с]
Фотосинтез - это процесс, при котором организм synthesирует свой собственный углеводы из углеводов, полученных от пищи.


Ответ короткий, как просили, но с ошибками: посреди русского текста вдруг английское «synthes…», а по смыслу
растения делают сахар не «из углеводов пищи», а из воды и углекислого газа. Модель видела мало русских текстов
и иногда «перескакивает» на язык, который знает лучше.

## 2. Как модель выбирает первое слово ответа

Заглянем внутрь, как в шаге 6: какие токены модель считает самыми вероятными в начале ответа?

In [5]:
for question in ["Какого цвета небо?", "Сколько будет 2 + 2?", "Как зовут автора «Евгения Онегина»?"]:
    text = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    with torch.no_grad():
        probs = model(**tok(text, return_tensors="pt")).logits[0, -1].softmax(-1)
    best = probs.topk(5)
    print(f"{question:40s} → " + ", ".join(f"{tok.decode([i])!r} {p:.0%}" for p, i in zip(best.values.tolist(), best.indices.tolist())))

Какого цвета небо?                       → 'Н' 30%, 'Как' 12%, 'В' 9%, 'На' 7%, 'К' 6%


Сколько будет 2 + 2?                     → '2' 79%, '1' 2%, 'Ч' 2%, 'В' 2%, 'И' 1%


Как зовут автора «Евгения Онегина»?      → 'Ав' 60%, 'И' 8%, 'В' 7%, '"' 3%, 'Р' 2%


## 3. Системная подсказка меняет поведение

Сообщение `system` — это инструкция, которую пользователь обычно не видит. Модель дообучали следовать ей.
Зададим один и тот же вопрос с разными инструкциями.

In [6]:
question = "Почему зимой холодно?"
for system in ["Ты — учитель физики. Объясни коротко и понятно, для пятиклассника.",
               "Ты — пират. Отвечай как пират, с пиратскими словечками.",
               "Отвечай одним предложением.",
               "Отвечай только словами «да» или «нет»."]:
    print(f"\n--- system: {system}")
    print(chat([{"role": "system", "content": system}, {"role": "user", "content": question}], max_new_tokens=120))


--- system: Ты — учитель физики. Объясни коротко и понятно, для пятиклассника.


[84 токенов за 5.7 с]
В зимнее время года холодно потому, что влажность воздуха уменьшается. Это происходит из-за того, что вода evaporates (выходит) из воды, когда она становится более тёплой. Влажность воздуха снижается, поэтому температура воздуха поднимается, что приводит к холодному дню.

--- system: Ты — пират. Отвечай как пират, с пиратскими словечками.


[55 токенов за 3.8 с]
В зимнее время года холодно из-за того, что вода в узком контейнере (море) не может проходить через небольшую длину, поэтому она быстро охлаждается и становится холодной.

--- system: Отвечай одним предложением.


[24 токенов за 1.6 с]
В зиме холоднее, потому что солнце не светит, а вода не нагревается.

--- system: Отвечай только словами «да» или «нет».


[4 токенов за 0.3 с]
Нет.


Инструкцию про **форму** ответа («одним предложением», «да или нет») модель выполняет: ответы стали короче.
А роль пирата не получилась — никаких пиратских словечек. И объяснение «учителя физики» — полная чепуха про влажность.
Маленькая модель хорошо подстраивает стиль и длину, но плохо знает факты.
А «Нет.» на вопрос «почему» — честно выполненная, но бессмысленная инструкция.

## 4. Определяем жанр подсказкой

В шаге 4 мы **учили** голову на 90 примерах и получили около 93% на 30 проверочных отрывках.
А что, если просто **попросить** чат-бота определить жанр? Это называется zero-shot — «без единого примера».

Второй вариант — few-shot: показать в подсказке по одному примеру каждого жанра.
Модель ничему не учится (веса не меняются) — она просто видит примеры в тексте и продолжает «по образцу».

In [7]:
rows = load_genres()
GENRES = list(dict.fromkeys(r["genre"] for r in rows))
test = [r for r in rows if r["split"] == "test"]
STEMS = {"сказ": "сказка", "детект": "детектив", "фантаст": "фантастика", "стих": "стихи", "новост": "новости",
         "научп": "научпоп", "научн": "научпоп"}

INSTRUCTION = ("Определи жанр текста. Ответь одним словом из списка: сказка, детектив, фантастика, стихи, новости, научпоп.\n"
               "(научпоп — научно-популярный текст)")


def parse_genre(answer):
    """Первый жанр, который встречается в ответе модели (или «?», если ни одного)."""
    answer = answer.lower()
    found = [(answer.find(stem), genre) for stem, genre in STEMS.items() if stem in answer]
    return min(found)[1] if found else "?"


def few_shot_messages(text):
    messages = [{"role": "system", "content": INSTRUCTION}]
    for g in GENRES:                                       # по одному обучающему примеру каждого жанра
        example = next(r["text"] for r in rows if r["genre"] == g and r["split"] == "train")
        messages += [{"role": "user", "content": example}, {"role": "assistant", "content": g}]
    return messages + [{"role": "user", "content": text}]


def zero_shot_messages(text):
    return [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": text}]


def genre_test(make, name, **chat_args):
    """Точность чат-бота на 30 проверочных отрывках + какие ответы он давал."""
    start = time.time()
    answers = [parse_genre(chat(make(r["text"]), max_new_tokens=6, show_time=False, **chat_args)) for r in test]
    acc = sum(a == r["genre"] for a, r in zip(answers, test)) / len(test)
    counts = {g: answers.count(g) for g in GENRES + ["?"] if answers.count(g)}
    print(f"{name}: точность {acc:.0%} ({time.time() - start:.0f} с на {len(test)} отрывков), ответы: {counts}")
    return acc


results = {"zero-shot": genre_test(zero_shot_messages, "zero-shot"),
           "few-shot": genre_test(few_shot_messages, "few-shot")}
print("для сравнения, голова из шага 4: 93%, наугад: 17%")

zero-shot: точность 17% (14 с на 30 отрывков), ответы: {'сказка': 30}


few-shot: точность 33% (34 с на 30 отрывков), ответы: {'сказка': 4, 'детектив': 7, 'фантастика': 6, 'научпоп': 10, '?': 3}
для сравнения, голова из шага 4: 93%, наугад: 17%


Маленький чат-бот справился **очень плохо**. Посмотрите на его ответы: без примеров он на **все** 30 отрывков отвечает «сказка».
Модель по сути не читает текст, а выдаёт один «любимый» ответ. По-русски модель на 0,5 млрд весов понимает плохо:
её учили в основном на английском и китайском (вспомните, как мелко она режет русские слова — шаг 1).
Примеры в подсказке немного помогают: ответы становятся разнообразнее, но точность всё равно низкая.

Попробуйте сами поменять порядок жанров в `INSTRUCTION` — изменится ли «любимый» ответ?

## 5. А если модель побольше?

Возьмём старшую сестру — **Qwen2.5-1.5B-Instruct**: то же устройство, но в 3 раза больше весов.
Цена: ~3 ГБ на диске, ~6 ГБ оперативной памяти и в 2–3 раза медленнее. Поэтому этот пункт выполняется,
только если модель уже скачана в `data/hf` (или если поставить `BIG_MODEL = True`).

Результаты одного запуска на 30 проверочных отрывках (на случай, если пункт пропущен):

| Модель | Весов | Жанр zero-shot | «Кто написал „Евгения Онегина“?» |
|--------|-------|----------------|-----------------------------------|
| Qwen2.5-0.5B | 0,5 млрд | 17% (всё — «сказка») | «Алексей Пушкин, 1867 год» |
| Qwen2.5-1.5B | 1,5 млрд | 67% | «А. С. Пушкин» |
| голова из шага 4 | 29 млн + 1 878 | 93% | — |

In [8]:
from common import is_downloaded

BIG_MODEL = None            # None — только если уже скачана; True — скачать (~3 ГБ); False — пропустить
if BIG_MODEL or (BIG_MODEL is None and is_downloaded("chat_big")):
    big_tok = load_tokenizer("chat_big")
    big_model = load_model("chat_big", "lm")
    results["1.5B zero-shot"] = genre_test(zero_shot_messages, "Qwen 1.5B, zero-shot", model=big_model, tok=big_tok)
    for question in ["Кто написал роман «Евгений Онегин»?", "Расскажи кратко о романе Пушкина «Капитан Немо»."]:
        print(f"\n>>> {question}\n{chat(question, max_new_tokens=80, model=big_model, tok=big_tok)}")
    del big_model
else:
    print("Пропускаем: модели Qwen2.5-1.5B нет в data/hf. Поставьте BIG_MODEL = True, чтобы скачать её (~3 ГБ).")

Qwen 1.5B, zero-shot: точность 67% (45 с на 30 отрывков), ответы: {'сказка': 3, 'детектив': 4, 'фантастика': 11, 'стихи': 4, 'новости': 5, 'научпоп': 2, '?': 1}


[35 токенов за 6.8 с]

>>> Кто написал роман «Евгений Онегин»?
Роман "Евгений Онегин" был написан великим русским писателем А.С. Пушкиным.


[80 токенов за 15.4 с]

>>> Расскажи кратко о романе Пушкина «Капитан Немо».
"Капитан Немо" - это французский детективный роман, написанный в 1863 году Александром Пушкиным. В книге повествуется о путешествии капитана Немо на корабль "Морской гном", который отправляется из Франции в Америку.

Главные геро


В 3 раза больше весов — и жанр угадывается в 4 раза чаще, а автор «Онегина» назван верно. Это и есть главный закон
языковых моделей последних лет: **чем больше модель и данных, тем лучше она понимает** (но тем она дороже и медленнее).
При этом и 1.5B уверенно сочиняет несуществующий роман «Капитан Немо» — от галлюцинаций размер полностью не спасает (пункт 7).

Сравните подходы к определению жанра:

| Подход | Нужны примеры? | Нужно обучение? | Скорость |
|--------|----------------|-----------------|----------|
| голова поверх BERT (шаг 4) | десятки | секунды | мгновенно |
| чат-бот, zero-shot | нет | нет | секунды на текст |
| чат-бот, few-shot | по одному в подсказке | нет | ещё медленнее (подсказка длиннее) |

Маленькая специализированная модель + 90 примеров обыграла универсальных чат-ботов в 17 и 50 раз больше её.
Зато чат-бота не нужно учить, и список жанров можно поменять одной строкой. Большие чат-боты
(в сотни раз больше Qwen) справились бы почти идеально.

## 6. Достаём данные из текста

Частая задача — превратить свободный текст в таблицу. Попросим модель вытащить из объявления данные в формате JSON
(это формат «ключ: значение», который легко прочитать программой).

In [9]:
ad = ("Внимание! В субботу, 14 декабря, в 11:00 в актовом зале школы № 5 пройдёт олимпиада по информатике "
      "для учеников 7–9 классов. Регистрация у Марии Ивановны в кабинете 203 до пятницы.")
answer = chat([
    {"role": "system", "content": "Извлеки из текста данные и ответь ТОЛЬКО JSON с ключами: событие, дата, время, место, кто_участвует, контакт."},
    {"role": "user", "content": ad},
], max_new_tokens=150)
print(answer)
try:
    data = json.loads(answer.strip("`").removeprefix("json").strip())
    print("\nJSON прочитан программой, например, дата:", data.get("дата"))
except json.JSONDecodeError:
    print("\nМодель ответила не чистым JSON — программа не может это прочитать. Так тоже бывает!")

[89 токенов за 6.8 с]
```json
{
    " события": [
        {
            "date": "2023-12-14",
            "time": "11:00",
            "location": "актовом зале школы № 5"
        }
    ],
    "who_участвует": ["Мария Ивановна"],
    "contact": "info@school.com"
}
```

JSON прочитан программой, например, дата: None


Посмотрите внимательно на ответ. Маленькая модель обычно:
- путает ключи (пишет их по-английски, хотя просили по-русски);
- теряет часть данных и путает роли (кто участвует, а кто принимает регистрацию);
- **добавляет то, чего в тексте нет**: год, адрес электронной почты. Это самое опасное — выглядит правдоподобно.

Программа, которая ждёт ключ `дата`, такой ответ не поймёт. Для подобных задач берут модели побольше
и проверяют ответ программой.

## 7. Галлюцинации: модель уверенно выдумывает

Модель обучена продолжать текст **правдоподобно**, а не **правдиво**. Если она не знает ответа, она всё равно
сочинит что-то похожее на ответ. Это называют **галлюцинациями**.

Спросим про то, чего не существует, и про то, что требует точности.

In [10]:
for question in [
    "Кто написал роман «Евгений Онегин»?",
    "Расскажи кратко о романе Пушкина «Капитан Немо».",            # такого романа нет
    "Кто такой знаменитый русский писатель Аркадий Пупырышкин?",   # и такого писателя нет
    "Сколько будет 37 × 43?",                                       # правильно: 1591
    "Сколько букв «р» в слове «программирование»?",                 # правильно: 3
]:
    print(f"\n>>> {question}\n{chat(question, max_new_tokens=100)}")

[87 токенов за 5.9 с]

>>> Кто написал роман «Евгений Онегин»?
Роман "Евгений Онегин" был написан в 1867 году автором Алексеем Пушкиным. Этот роман является одним из самых известных произведений писателя и один из самых популярных в истории литературы. Он был опубликован в 1869 году и продолжает быть актуальным на сегодняшний день.


[100 токенов за 7.2 с]

>>> Расскажи кратко о романе Пушкина «Капитан Немо».
"Капитан Немо" - это одно из самых известных произведений писателя Александр Пушкина. Этот роман был написан в 1790 году и представляет собой продолжение его знаменитого романа "Маленький Капитан". 

В романе главный герой, капитан Немо, путешествует по морю и сталкивается с различными опасностями, включая


[100 токенов за 6.9 с]

>>> Кто такой знаменитый русский писатель Аркадий Пупырышкин?
Аркадий Пупырышкин - известный российский писатель и переводчик, который известен своими работами по истории и культуре России. Вот некоторые ключевые факты о нем:

1. Начало жизни: Родился в 1938 году в Москве.

2. Ведение творчества: Публикует более 50 книг, многие из которых были переведены на


[30 токенов за 2.1 с]

>>> Сколько будет 37 × 43?
37 multiplied by 43 is:

1,591

This calculation can be done using the standard multiplication method or a calculator.


[100 токенов за 6.8 с]

>>> Сколько букв «р» в слове «программирование»?
В слове "программирование" буква "р" встречается 2 раза.

Почему? Это связано с тем, что "программирование" состоит из двух частей: "программа" и "разработка". В каждой части есть одна буква "р".

Если мы рассмотрим каждую часть:

1. "Программа" - это слово, которое начинается с буквы "пр".
2. "Р


**Обсудим:**
- даже на известный вопрос маленькая модель отвечает с ошибками: Пушкин у неё «Алексей», а роман написан «в 1867 году»
  (на самом деле Александр Сергеевич, 1823–1831). Связь «Онегин → Пушкин» она помнит, а детали — нет;
- про несуществующий роман и писателя модель сочиняет правдоподобную историю вместо «я не знаю»;
- умножение 37 × 43 = 1591 она решила верно (такие примеры часто встречаются в обучающих текстах), но ответила по-английски;
- с подсчётом букв беда: модель видит **токены**, а не буквы (шаг 1), и «считает» угадыванием, а потом ещё и придумывает объяснение;
- **главный вывод:** ответу чат-бота нельзя слепо верить. Большие модели ошибаются реже, но тоже ошибаются — важные факты нужно проверять.

## 8. Свой диалог

Диалог с памятью — это просто список сообщений, который растёт: каждая новая реплика добавляется в конец,
и **весь** диалог заново отдаётся модели. Модель ничего не «помнит» между вызовами — помнит список.

In [11]:
dialog = [{"role": "system", "content": "Ты — дружелюбный помощник. Отвечай кратко, по-русски."}]
for question in ["Меня зовут Маша. Я люблю астрономию.", "Посоветуй, что почитать по моему увлечению.", "Как меня зовут?"]:
    dialog.append({"role": "user", "content": question})
    answer = chat(dialog, max_new_tokens=120)
    dialog.append({"role": "assistant", "content": answer})
    print(f"\n>>> {question}\n{answer}")
print("\nсообщений в диалоге:", len(dialog), "— всё это модель читает заново при каждом ответе")

[14 токенов за 1.1 с]

>>> Меня зовут Маша. Я люблю астрономию.
Привет! Как я могу вам помочь сегодня?


[120 токенов за 8.4 с]

>>> Посоветуй, что почитать по моему увлечению.
Вот несколько книг, которые могут быть полезными для вас:

1. "Астрономия" Джеймса Кеннеди
2. "Космическая эра" Стивена Уолша
3. "Солнце и звезды" Роберт Бернардес
4. "История космонавтики" Эндрю Гарри
5. "Космический манускрипт" Джошуа Льюис

Эти книги обзорные и


[11 токенов за 1.1 с]

>>> Как меня зовут?
Вы можете называть меня просто Маша.

сообщений в диалоге: 7 — всё это модель читает заново при каждом ответе


Модель «помнит», что пользователь любит астрономию, — потому что эта реплика лежит в списке. Но обратите внимание:
- книги, которые она советует, скорее всего, **выдуманы** (проверьте названия и авторов в интернете!);
- на «Как меня зовут?» модель может ответить невпопад — перепутать, кто есть кто в диалоге.

## Попробуйте сами

1. Придумайте свою системную подсказку: «Ты — Шерлок Холмс», «Ты — экскурсовод по Москве». Насколько хорошо модель держит роль?
2. Найдите вопрос, на который модель отвечает неправильно. Можно ли получить верный ответ, изменив формулировку?
3. Поменяйте `INSTRUCTION` в пункте 4 (например, добавьте описание каждого жанра). Вырастет ли точность?
4. Попросите модель написать четверостишие про осень и прогоните его через голову жанров из шага 4.
5. Включите температуру (`chat(..., temperature=1.0)`) и задайте один вопрос 3 раза. Одинаковые ли ответы?